# 03 - Leaf Segmentation (OpenCV Background Removal)

Leaf isolation for field/uncontrolled photos: hand, soil, other leaves, and
other branches in the background need to be removed BEFORE the crop
identifier ever sees the image.

Strategy: pure color thresholding alone can't tell "the leaf you're
photographing" apart from "other green leaves in the background" -- but two
extra cues help a lot:
  1. FOCUS: farmers instinctively focus the camera on the leaf they're
     photographing, so background leaves/branches are usually slightly
     out of focus (lower local sharpness).
  2. POSITION/SIZE: the subject leaf is almost always the largest and most
     central green region in the frame.

This script combines all three (color + sharpness + position/size) into a
single score per candidate region, picks the winner, and crops it out with
padding. Use this same function both to (a) generate your own leaf-crop
training set from the raw "Uncontrolled Environment" whole-plant photos,
replacing your reliance on the pre-made "Created Dataset" crops, and
(b) as the first stage of your live inference pipeline.

Install deps:
    pip install opencv-python numpy --break-system-packages

## Imports & Configuration

In [1]:
import cv2
import numpy as np
from pathlib import Path

## `compute_sharpness_map`

Local sharpness via a sliding-window Laplacian variance map.

In [2]:
def compute_sharpness_map(gray, ksize=25):
    """Local sharpness via a sliding-window Laplacian variance map."""
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    lap_sq = lap ** 2
    local_mean = cv2.blur(lap_sq, (ksize, ksize))
    return local_mean  # higher = sharper/more in-focus

## `green_mask`

Broad green-vegetation mask. Deliberately wide range since leaf color

In [3]:
def green_mask(hsv):
    """
    Broad green-vegetation mask. Deliberately wide range since leaf color
    varies with disease/lighting -- we rely on sharpness+position to pick
    the RIGHT green region, not on color alone to exclude every other
    green thing.
    """
    lower = np.array([25, 30, 30])
    upper = np.array([95, 255, 255])
    mask = cv2.inRange(hsv, lower, upper)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    return mask

## `score_contour`

Score a candidate leaf region on:

In [4]:
def score_contour(contour, image_shape, sharpness_map, mask):
    """
    Score a candidate leaf region on:
      - area (bigger = more likely the subject, normalized by image size)
      - centrality (closer to image center = more likely the subject)
      - mean sharpness inside the region (in-focus = more likely the subject)
    Returns a single combined score; higher is better.
    """
    h, w = image_shape[:2]
    area = cv2.contourArea(contour)
    if area < 0.005 * h * w:  # ignore tiny specks
        return -1, None

    x, y, bw, bh = cv2.boundingRect(contour)
    cx, cy = x + bw / 2, y + bh / 2
    img_cx, img_cy = w / 2, h / 2
    dist_from_center = np.hypot(cx - img_cx, cy - img_cy)
    max_dist = np.hypot(img_cx, img_cy)
    centrality_score = 1 - (dist_from_center / max_dist)

    region_mask = np.zeros((h, w), dtype=np.uint8)
    cv2.drawContours(region_mask, [contour], -1, 255, thickness=cv2.FILLED)
    mean_sharpness = cv2.mean(sharpness_map, mask=region_mask)[0]

    area_score = area / (h * w)

    # Weighted combination -- tune these weights against a few validation
    # images if the wrong leaf keeps winning.
    combined = (0.45 * area_score) + (0.30 * centrality_score) + (0.25 * min(mean_sharpness / 500, 1.0))
    return combined, (x, y, bw, bh)

## `isolate_subject_leaf`

Returns the cropped subject-leaf image (BGR), or None if no confident

In [5]:
def isolate_subject_leaf(image_bgr, padding_ratio=0.08):
    """
    Returns the cropped subject-leaf image (BGR), or None if no confident
    candidate was found (caller should flag this for manual review / ask
    the farmer to retake the photo).
    """
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)

    sharpness_map = compute_sharpness_map(gray)
    mask = green_mask(hsv)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None

    best_score, best_box = -1, None
    for c in contours:
        score, box = score_contour(c, image_bgr.shape, sharpness_map, mask)
        if score > best_score:
            best_score, best_box = score, box

    if best_box is None or best_score < 0.15:  # confidence floor
        return None

    x, y, bw, bh = best_box
    pad_x, pad_y = int(bw * padding_ratio), int(bh * padding_ratio)
    h, w = image_bgr.shape[:2]
    x0, y0 = max(0, x - pad_x), max(0, y - pad_y)
    x1, y1 = min(w, x + bw + pad_x), min(h, y + bh + pad_y)

    return image_bgr[y0:y1, x0:x1]

## `process_folder`

Batch-processes a folder of raw field photos: saves successfully

In [6]:
def process_folder(src_dir, dst_dir, review_dir):
    """
    Batch-processes a folder of raw field photos: saves successfully
    isolated leaf crops to dst_dir, and copies anything with no confident
    candidate to review_dir for manual inspection instead of silently
    dropping it or feeding a bad crop into training.
    """
    src_dir, dst_dir, review_dir = Path(src_dir), Path(dst_dir), Path(review_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)
    review_dir.mkdir(parents=True, exist_ok=True)

    for img_path in src_dir.rglob("*"):
        if img_path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue
        image = cv2.imread(str(img_path))
        if image is None:
            continue
        crop = isolate_subject_leaf(image)
        if crop is None or crop.size == 0:
            cv2.imwrite(str(review_dir / img_path.name), image)
        else:
            cv2.imwrite(str(dst_dir / img_path.name), crop)

## Run

In [7]:
# Example: regenerate Groundnut leaf crops yourself from the raw
# Uncontrolled Environment photos, instead of relying on the pre-made
# "Created Dataset" folder.
process_folder(
    src_dir=r"G:\Crop Identification\Final Datasets\Groundnut Leaf Uncontrolled Environment",
    dst_dir="generated_crops/groundnut/leaf_crops",
    review_dir="generated_crops/groundnut/needs_review",
)